Pre-requisite:

    I used code from the lecture to generate 2 Bitcoin testnet addresses and multiSig address:

===============================
First address data
================================
+----------------------------+------------------------------------------------------------------------------------------------------------------------------------+
|        Discription         |                                                               Value                                                                |
+----------------------------+------------------------------------------------------------------------------------------------------------------------------------+
|      Private Key, hex      |                                  c085aabe15e4964544da042611c8b25b95bf0aae843a39a7c2d29abdd9edda7e                                  |
|      Public Key, hex       | 04d85c73109dd13ad653da9c6ed007953e7055338a8f80f9fe164a86cc3907ad2abc546315424c90f680abf9ac4d9289623700998ec71d6084308ac00e407b40e0 |
| Compressed Public Key, hex |                                 02d85c73109dd13ad653da9c6ed007953e7055338a8f80f9fe164a86cc3907ad2a                                 |
|  MainNet Address, base58   |                                                 1H3Sevje6gCYHineydkCkLqi8YyaiyekYc                                                 |
|  Testnet Address, base58   |                                                 mwZPwypcuhdo4qGGhCiaaG42zYaHeUBbVU                                                 |
+----------------------------+------------------------------------------------------------------------------------------------------------------------------------+

===============================
Second address data
================================
+----------------------------+------------------------------------------------------------------------------------------------------------------------------------+
|        Discription         |                                                               Value                                                                |
+----------------------------+------------------------------------------------------------------------------------------------------------------------------------+
|      Private Key, hex      |                                  0c44ebefb10ff08e50104ed1689e4fb7f2e5806b83231f17d2913e562235e376                                  |
|      Public Key, hex       | 040afa0b67a880dd1569ef95d699c5de44c3aad9b585212f6bf5ada57bed0555477386edf63a434eed3701792935433d1534e734f41556b001541324780007e9cb |
| Compressed Public Key, hex |                                 030afa0b67a880dd1569ef95d699c5de44c3aad9b585212f6bf5ada57bed055547                                 |
|  MainNet Address, base58   |                                                 1Q7ZPiP4L4qQkqXBSjXiefR7YicQsrrhmg                                                 |
|  Testnet Address, base58   |                                                 n4dWgmU396GfXwzoAJW6UadSQiD7gcCWdL                                                 |
+----------------------------+------------------------------------------------------------------------------------------------------------------------------------+


MultiSig address  2MxA4TfF1ZntNMojeR1Lg1Exko29oM4LiPr

Redeem script 

524104d85c73109dd13ad653da9c6ed007953e7055338a8f80f9fe164a86cc3907ad2abc546315424c90f680abf9ac4d9289623700998ec71d6084308ac00e407b40e041040afa0b67a880dd1569ef95d699c5de44c3aad9b585212f6bf5ada57bed0555477386edf63a434eed3701792935433d1534e734f41556b001541324780007e9cb52ae

1 task:

In [2]:
address_1 = "mwZPwypcuhdo4qGGhCiaaG42zYaHeUBbVU"
address_2 = "n4dWgmU396GfXwzoAJW6UadSQiD7gcCWdL"
multiSig_address = "2MxA4TfF1ZntNMojeR1Lg1Exko29oM4LiPr"
redeem_script = "524104d85c73109dd13ad653da9c6ed007953e7055338a8f80f9fe164a86cc3907ad2abc546315424c90f680abf9ac4d9289623700998ec71d6084308ac00e407b40e041040afa0b67a880dd1569ef95d699c5de44c3aad9b585212f6bf5ada57bed0555477386edf63a434eed3701792935433d1534e734f41556b001541324780007e9cb52ae"
private_key_1 = "c085aabe15e4964544da042611c8b25b95bf0aae843a39a7c2d29abdd9edda7e"
private_key_2 = "0c44ebefb10ff08e50104ed1689e4fb7f2e5806b83231f17d2913e562235e376"
public_key_1 = "04d85c73109dd13ad653da9c6ed007953e7055338a8f80f9fe164a86cc3907ad2abc546315424c90f680abf9ac4d9289623700998ec71d6084308ac00e407b40e0"
public_key_2 = "040afa0b67a880dd1569ef95d699c5de44c3aad9b585212f6bf5ada57bed0555477386edf63a434eed3701792935433d1534e734f41556b001541324780007e9cb"


2 task:

In [20]:
print("c7d263153b654943b20d43c2b7ba3d79bf4d571ba3dac32cfe5ffc3869ba5b22")


c7d263153b654943b20d43c2b7ba3d79bf4d571ba3dac32cfe5ffc3869ba5b22


3 task:

In [ ]:
import hashlib
from ecdsa import SigningKey, SECP256k1
from ecdsa.util import sigencode_der_canonize


prev_txid_hex = "c7d263153b654943b20d43c2b7ba3d79bf4d571ba3dac32cfe5ffc3869ba5b22"
prev_vout = 1
prev_amount_sat = 181597


fee_sat = 500
to_addr2_sat = 100000
to_multisig_sat = prev_amount_sat - fee_sat - to_addr2_sat
assert to_multisig_sat > 0, "Not enough funds for planned outputs + fee"


B58_ALPH = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"
def b58decode_check(addr: str) -> bytes:
    n = 0
    for c in addr:
        n = n * 58 + B58_ALPH.index(c)
    full = n.to_bytes((n.bit_length() + 7) // 8, "big")
    pad = 0
    for c in addr:
        if c == "1":
            pad += 1
        else:
            break
    full = b"\x00" * pad + full
    payload, checksum = full[:-4], full[-4:]
    chk = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
    if chk != checksum:
        raise ValueError("Bad base58 checksum")
    return payload

def hash256(b: bytes) -> bytes:
    return hashlib.sha256(hashlib.sha256(b).digest()).digest()

def hash160(b: bytes) -> bytes:
    return hashlib.new("ripemd160", hashlib.sha256(b).digest()).digest()

def le_u32(i: int) -> bytes:
    return i.to_bytes(4, "little")

def le_u64(i: int) -> bytes:
    return i.to_bytes(8, "little")

def varint(n: int) -> bytes:
    if n < 0xfd:
        return bytes([n])
    if n <= 0xffff:
        return b"\xfd" + n.to_bytes(2, "little")
    if n <= 0xffffffff:
        return b"\xfe" + n.to_bytes(4, "little")
    return b"\xff" + n.to_bytes(8, "little")

def p2pkh_scriptpubkey_from_address(addr: str) -> bytes:
    payload = b58decode_check(addr)  # version + h160
    h160 = payload[1:]
    return b"\x76\xa9\x14" + h160 + b"\x88\xac"

def p2sh_scriptpubkey_from_redeemscript(redeem_hex: str) -> bytes:
    rs = bytes.fromhex(redeem_hex)
    h160 = hash160(rs)
    return b"\xa9\x14" + h160 + b"\x87"

def txid_from_raw(raw: bytes) -> str:
    return hash256(raw)[::-1].hex()


version = le_u32(1)
locktime = le_u32(0)
sighash_all = le_u32(1)

prev_txid_le = bytes.fromhex(prev_txid_hex)[::-1]
outpoint = prev_txid_le + le_u32(prev_vout)

sequence = b"\xff\xff\xff\xff"

script_pubkey_prev = p2pkh_scriptpubkey_from_address(address_1)

script_pubkey_addr2 = p2pkh_scriptpubkey_from_address(address_2)
script_pubkey_multisig = p2sh_scriptpubkey_from_redeemscript(redeem_script)

vout = (
    le_u64(to_addr2_sat) + varint(len(script_pubkey_addr2)) + script_pubkey_addr2 +
    le_u64(to_multisig_sat) + varint(len(script_pubkey_multisig)) + script_pubkey_multisig
)


vin_for_sig = (
    varint(1) +
    outpoint +
    varint(len(script_pubkey_prev)) + script_pubkey_prev +
    sequence
)
preimage = version + vin_for_sig + varint(2) + vout + locktime + sighash_all
z = hash256(preimage)


sk = SigningKey.from_string(bytes.fromhex(private_key_1), curve=SECP256k1)
sig_der = sk.sign_digest(z, sigencode=sigencode_der_canonize)
sig = sig_der + b"\x01"

pubkey = bytes.fromhex(public_key_1)


def push(data: bytes) -> bytes:
    l = len(data)
    if l < 0x4c:
        return bytes([l]) + data
    raise ValueError("Data too long for simple push")

scriptSig = push(sig) + push(pubkey)

vin_final = (
    varint(1) +
    outpoint +
    varint(len(scriptSig)) + scriptSig +
    sequence
)

raw_tx = version + vin_final + varint(2) + vout + locktime

print("Task 3 raw tx hex:\n", raw_tx.hex())
print("Task 3 txid:\n", txid_from_raw(raw_tx))
print("Outputs:")
print("  to Address_2:", to_addr2_sat, "sat ->", address_2)
print("  to MultiSig :", to_multisig_sat, "sat ->", multiSig_address)
print("  fee         :", fee_sat, "sat")

Task 3 raw tx hex:
 0100000001225bba6938fc5ffe2cc3daa31b574dbf793dbab7c2430db24349653b1563d2c7010000008a473044022076ae3b9e58f00f66eb24569068bde480c11d8ce14e33e9f9eae5b85a59531fc1022035c9aa0e49ae179e89e80d0571c1a825a009af3a9ebfa29dae734acb98ed175b014104d85c73109dd13ad653da9c6ed007953e7055338a8f80f9fe164a86cc3907ad2abc546315424c90f680abf9ac4d9289623700998ec71d6084308ac00e407b40e0ffffffff02a0860100000000001976a914fd886d8e5450cee3d6b758499e85fc606bf423e488acc93c01000000000017a91435de98753e17810d6d5ca3594cd6cbecb5dd8a1b8700000000
Task 3 txid:
 9428c7ed20a9d66dfd0c88b4861a4420b873faaf516e28ba40abcf424f49b005
Outputs:
  to Address_2: 100000 sat -> n4dWgmU396GfXwzoAJW6UadSQiD7gcCWdL
  to MultiSig : 81097 sat -> 2MxA4TfF1ZntNMojeR1Lg1Exko29oM4LiPr
  fee         : 500 sat


Transaction_ID 9428c7ed20a9d66dfd0c88b4861a4420b873faaf516e28ba40abcf424f49b005

4 task:

In [ ]:
prev_txid_hex = "5fa722255c1e9f8b53457bcf3f0e3a2d15179fd2d0dd6f73e20e4795de354c67"
prev_vout = 0
prev_amount_sat = 153026

fee_sat = 800 
op_return_text = "Baronov Nikita"



def op_return_script(data: bytes) -> bytes:
    if len(data) > 80:
        raise ValueError("OP_RETURN data too long (keep <= 80 bytes)")
    return b"\x6a" + bytes([len(data)]) + data


if prev_amount_sat <= 0:
    raise ValueError("Fill prev_amount_sat with the UTXO value in satoshis (must be > 0).")

change_sat = prev_amount_sat - fee_sat
if change_sat < 546:
    raise ValueError(f"Change {change_sat} sat is dust. Use a larger UTXO or reduce fee.")

version = le_u32(1)
locktime = le_u32(0)
sighash_all = le_u32(1)

prev_txid_le = bytes.fromhex(prev_txid_hex)[::-1]
outpoint = prev_txid_le + le_u32(prev_vout)
sequence = b"\xff\xff\xff\xff"

script_pubkey_prev = p2pkh_scriptpubkey_from_address(address_1)

data_bytes = op_return_text.encode("utf-8")
spk_opret = op_return_script(data_bytes)
vout0 = le_u64(0) + varint(len(spk_opret)) + spk_opret

spk_change = p2pkh_scriptpubkey_from_address(address_1)
vout1 = le_u64(change_sat) + varint(len(spk_change)) + spk_change

vout = vout0 + vout1

vin_for_sig = (
    varint(1) +
    outpoint +
    varint(len(script_pubkey_prev)) + script_pubkey_prev +
    sequence
)
preimage = version + vin_for_sig + varint(2) + vout + locktime + sighash_all
z = hash256(preimage)

sk = SigningKey.from_string(bytes.fromhex(private_key_1), curve=SECP256k1)
sig_der = sk.sign_digest(z, sigencode=sigencode_der_canonize)
sig = sig_der + b"\x01"

pubkey = bytes.fromhex(public_key_1)
scriptSig = push(sig) + push(pubkey)

vin_final = (
    varint(1) +
    outpoint +
    varint(len(scriptSig)) + scriptSig +
    sequence
)

raw_tx = version + vin_final + varint(2) + vout + locktime

print("Step 3.1 OP_RETURN raw tx hex:\n", raw_tx.hex())
print("Step 3.1 txid:\n", txid_from_raw(raw_tx))
print("Embedded data:", op_return_text)
print("Change back to address_1:", change_sat, "sat ->", address_1)
print("Fee:", fee_sat, "sat")

[{'txid': '5fa722255c1e9f8b53457bcf3f0e3a2d15179fd2d0dd6f73e20e4795de354c67', 'vout': 0, 'status': {'confirmed': True, 'block_height': 4806186, 'block_hash': '0000000008cb7ed1971e83bbc646eca9224c340a238922b6ef1c1df970b96015', 'block_time': 1765485934}, 'value': 153026}]
Step 3.1 OP_RETURN raw tx hex:
 0100000001674c35de95470ee2736fddd0d29f17152d3a0e3fcf7b45538b9f1e5c2522a75f000000008a47304402201039cf71e3264a36862ce349889afc5e15d00fc31cf6bb017cebf313fd7ddfc6022036056ed50887f8760d715b5f749b30f1cdeef9c0681aad0dfec191f1c47a8f71014104d85c73109dd13ad653da9c6ed007953e7055338a8f80f9fe164a86cc3907ad2abc546315424c90f680abf9ac4d9289623700998ec71d6084308ac00e407b40e0ffffffff020000000000000000106a0e4261726f6e6f76204e696b697461a2520200000000001976a914aff8425f339d2b03925415c0c00451c5696f940688ac00000000
Step 3.1 txid:
 f7f7167eef65e87abd90df52e72f8ec2d53b569d066d248126ac2b07cde67a24
Embedded data: Baronov Nikita
Change back to address_1: 152226 sat -> mwZPwypcuhdo4qGGhCiaaG42zYaHeUBbVU
Fee: 800 sat


Transaction_ID f7f7167eef65e87abd90df52e72f8ec2d53b569d066d248126ac2b07cde67a24

5 task:

In [23]:
ADDRESS_TARGET = "n2jmkNzxLJM51AvH3WNyYgWpsrjvR5HnF8"


prev_txid_hex = "f7f7167eef65e87abd90df52e72f8ec2d53b569d066d248126ac2b07cde67a24"
prev_vout = 1  
prev_amount_sat = 100000  


fee_sat = 500
send_amount_sat = prev_amount_sat - fee_sat

assert send_amount_sat >= 546, f"Сумма отправки {send_amount_sat} сатоши меньше минимума 546"




version = le_u32(1)
locktime = le_u32(0)
sighash_all = le_u32(1)

prev_txid_le = bytes.fromhex(prev_txid_hex)[::-1]
outpoint = prev_txid_le + le_u32(prev_vout)
sequence = b"\xff\xff\xff\xff"

script_pubkey_prev = p2pkh_scriptpubkey_from_address(address_1)


script_pubkey_target = p2pkh_scriptpubkey_from_address(ADDRESS_TARGET)
vout = le_u64(send_amount_sat) + varint(len(script_pubkey_target)) + script_pubkey_target


vin_for_sig = (
    varint(1) +
    outpoint +
    varint(len(script_pubkey_prev)) + script_pubkey_prev +
    sequence
)
preimage = version + vin_for_sig + varint(1) + vout + locktime + sighash_all
z = hash256(preimage)


sk = SigningKey.from_string(bytes.fromhex(private_key_1), curve=SECP256k1)
sig_der = sk.sign_digest(z, sigencode=sigencode_der_canonize)
sig = sig_der + b"\x01"

pubkey = bytes.fromhex(public_key_1)
scriptSig = push(sig) + push(pubkey)

vin_final = (
    varint(1) +
    outpoint +
    varint(len(scriptSig)) + scriptSig +
    sequence
)


raw_tx = version + vin_final + varint(1) + vout + locktime


print("="*60)
print("FINAL TRANSACTION (Step 3.2)")
print("="*60)
print("Raw tx hex:")
print(raw_tx.hex())
print("\nTXID (calculated):")
print(txid_from_raw(raw_tx))
print("\nDetails:")
print(f"  Input UTXO: {prev_txid_hex}:{prev_vout}")
print(f"  Input amount: {prev_amount_sat} sat")
print(f"  Sent to {ADDRESS_TARGET}: {send_amount_sat} sat")
print(f"  Fee: {fee_sat} sat")
print(f"  Change: 0 sat (all sent)")
print("="*60)

FINAL TRANSACTION (Step 3.2)
Raw tx hex:
0100000001247ae6cd072bac2681246d069d563bd5c28e2fe752df90bd7ae865ef7e16f7f7010000008a47304402205979d78b17a7d414953f5a93360c487cac48a19648ca710e487fa3902d4358d8022071fcbae47c14ee17ec862241a9afa003db83b4567be0284392ae942de0ebfb99014104d85c73109dd13ad653da9c6ed007953e7055338a8f80f9fe164a86cc3907ad2abc546315424c90f680abf9ac4d9289623700998ec71d6084308ac00e407b40e0ffffffff01ac840100000000001976a914e8c73eda67ba6bc8b2d52b1d60da9aad9457f6f588ac00000000

TXID (calculated):
e3383b583af54d7976f2ef2b586188ede51a366b77cb3302793475b0790b7f63

Details:
  Input UTXO: f7f7167eef65e87abd90df52e72f8ec2d53b569d066d248126ac2b07cde67a24:1
  Input amount: 100000 sat
  Sent to n2jmkNzxLJM51AvH3WNyYgWpsrjvR5HnF8: 99500 sat
  Fee: 500 sat
  Change: 0 sat (all sent)


Transaction_ID e3383b583af54d7976f2ef2b586188ede51a366b77cb3302793475b0790b7f63